# Week 4: Hypothesis Testing — India GDP
**H1:** Welch's two-sample t-test — did mean GDP growth change between 2001-2010 and 2011-2020?

**H2:** Regression slope t-test — is the long-run GDP growth trend (2000-2025) statistically significant?

Upload `gdp_dataset.csv` to the Colab file browser before running.

In [ ]:
# Install/import required libraries (all pre-installed on Colab)import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy import stats

In [ ]:
# ---------------------------------------------------------------# 1. DATA — India GDP dataset (World Bank, GDP current US$, 2000-2025)# ---------------------------------------------------------------# If running fresh in Colab, first upload the World Bank CSV and# rebuild this dataframe as in the Week 2 notebook, or upload# gdp_dataset.csv directly:# from google.colab import files# files.upload()df = pd.read_csv("gdp_dataset.csv")df.tail(10)

In [ ]:
# ---------------------------------------------------------------# 2. HYPOTHESIS 1 — Growth rate: 2001-2010 vs 2011-2020# H0: mean growth is equal in both periods# H1: mean growth differs (two-tailed)# ---------------------------------------------------------------g = df.dropna(subset=["Growth_%"])period1 = g[(g.Year >= 2001) & (g.Year <= 2010)]["Growth_%"].valuesperiod2 = g[(g.Year >= 2011) & (g.Year <= 2020)]["Growth_%"].valuesprint(f"2001-2010: n={len(period1)}, mean={period1.mean():.2f}%, std={period1.std(ddof=1):.2f}%")print(f"2011-2020: n={len(period2)}, mean={period2.mean():.2f}%, std={period2.std(ddof=1):.2f}%")

In [ ]:
# Levene's test for equal variances (informs t-test type)levene_stat, levene_p = stats.levene(period1, period2)print(f"Levene's test: stat={levene_stat:.3f}, p={levene_p:.4f}")# Welch's two-sample t-test (unequal variances)t_stat, p_value = stats.ttest_ind(period1, period2, equal_var=False)print(f"Welch's t-test: t={t_stat:.3f}, p={p_value:.4f}")alpha = 0.05print("Reject H0" if p_value < alpha else "Fail to reject H0")

In [ ]:
# Boxplot comparisonplt.style.use("seaborn-v0_8-whitegrid")fig, ax = plt.subplots(figsize=(7.5, 5))bp = ax.boxplot([period1, period2], tick_labels=["2001-2010", "2011-2020"],                 patch_artist=True, widths=0.5)for patch, color in zip(bp['boxes'], ["#1f4e79", "#e07b00"]):    patch.set_facecolor(color); patch.set_alpha(0.6)ax.set_ylabel("Annual GDP Growth (%)")ax.set_title("GDP Growth Rate Distribution: 2001-2010 vs 2011-2020")plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------# 3. HYPOTHESIS 2 — Significance of the long-run GDP growth trend# H0: slope of ln(GDP) vs Year = 0 (no trend)# H1: slope != 0 (significant trend)# ---------------------------------------------------------------years = df["Year"].values.astype(float)log_gdp = np.log(df["GDP_USD_Billion"].values)slope, intercept, r_value, p_value_slope, std_err = stats.linregress(years, log_gdp)t_stat_slope = slope / std_errcagr = (np.exp(slope) - 1) * 100print(f"slope={slope:.5f}, t={t_stat_slope:.3f}, p={p_value_slope:.6f}, R2={r_value**2:.4f}")print(f"Implied CAGR: {cagr:.2f}%")print("Reject H0" if p_value_slope < alpha else "Fail to reject H0")

In [ ]:
# Regression plotfig, ax = plt.subplots(figsize=(8.5, 5))ax.scatter(years, log_gdp, color="#1f4e79", zorder=3, label="Observed ln(GDP)")ax.plot(years, intercept + slope*years, color="#c00000", linewidth=2,        label=f"Fitted trend (slope={slope:.4f}, p<0.001)")ax.set_xlabel("Year"); ax.set_ylabel("ln(GDP, USD Billion)")ax.set_title("Log-GDP Trend Regression, 2000-2025 (Test of H0: slope = 0)")ax.legend()plt.tight_layout(); plt.show()